# 02 — Compile formats (admin / public wedge)

ClearMetric emits the **same enforced graph** in several shapes. Admin formats
(public wedge) are raw JSON — no consumer envelope, no `--identity`.

| Format | Purpose |
|--------|--------|
| `json` | Full graph artifact |
| `catalog` | table / column / model slice for catalog UIs |
| `openlineage` | Interop with DataHub, Marquez, etc. |
| `text` | Human-readable summary |

In [ ]:
import json
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _paths import wedge_project

PROJECT_DIR = wedge_project()

In [ ]:
from clearmetric.compiler import compile
from clearmetric.emitters import emit_compile
from clearmetric.emitters.registry import WEDGE_COMPILE_FORMATS

compiled = compile(PROJECT_DIR)
print("public wedge formats:", WEDGE_COMPILE_FORMATS)

## Full graph (`json`)

In [ ]:
graph_json = emit_compile("json", compiled)
graph = json.loads(graph_json)
print(f"nodes: {len(graph['nodes'])}  edges: {len(graph['edges'])}")
print("sample node ids:", [n["id"] for n in graph["nodes"][:5]])

## Catalog slice (`catalog`)

Metrics and queries from intent are **excluded** from catalog — only physical assets.

In [ ]:
catalog_json = emit_compile("catalog", compiled)
catalog = json.loads(catalog_json)
catalog_kinds = {node["kind"] for node in catalog["nodes"]}
graph_kinds = {node["kind"] for node in graph["nodes"]}

print(f"catalog nodes: {len(catalog['nodes'])}")
print(f"catalog kinds: {sorted(catalog_kinds)}")
print(f"kinds only in full graph: {sorted(graph_kinds - catalog_kinds)}")

## OpenLineage export

Returns a JSON object (not NDJSON) suitable for lineage platform ingestion.

In [ ]:
ol_json = emit_compile("openlineage", compiled)
ol = json.loads(ol_json)
print("top-level keys:", sorted(ol.keys()))
events = ol.get("events") or []
print(f"events: {len(events)}")
if events:
    print("first event keys:", sorted(events[0].keys()))

## Validate emitted JSON against schema

Validation lives in `clearmetric.core.validate` — single source of truth for artifacts.

In [ ]:
from clearmetric.core.validate import validate_artifact_dict

validate_artifact_dict(catalog)
print("catalog artifact passes catalog-artifact.schema.json")